In [ ]:
import pandas as pd

In [ ]:
def read_dict(filename):
    d = dict()
    with open(filename, "r") as f:
        for line in f:
            line = line.strip().split("\t")
            d[line[0]] = int(line[1])
    return d
def write_dict(filename, d):
    with open(filename, "w") as f:
        for k, v in d.items():
            f.write(k)
            f.write("\t")
            f.write(str(v))
            f.write("\n")

In [ ]:
df1 = pd.read_csv("~/Downloads/data/WK3l-15k/en_de/p_en_v6.csv", sep="@@@", header=None)
df1.columns = ["head", "relation", "tail"]
df2 = pd.read_csv("~/Downloads/data/WK3l-15k/en_fr/p_en_v5.csv", sep="@@@", header=None)
df2.columns = ["head", "relation", "tail"]

In [ ]:
df = pd.merge(df1, df2, how="outer")

In [ ]:
df1.shape, df2.shape, df.shape

In [ ]:
df

In [ ]:
df["head"] = df["head"].apply(str)
df["relation"] = df["relation"].apply(str)
df["tail"] = df["tail"].apply(str)

entities = sorted(set(df["head"]) | set(df["tail"]))
relations = sorted(set(df["relation"]))

entity_dict = {x: i for i, x in enumerate(entities)}
relation_dict = {x: i for i, x in enumerate(relations)}

df["head"] = df["head"].apply(lambda x: entity_dict.get(x, -1))
df["tail"] = df["tail"].apply(lambda x: entity_dict.get(x, -1))
df["relation"] = df["relation"].apply(lambda x: relation_dict.get(x, -1))

write_dict("../data/WK3l-15k_EN/entity_dict.txt", entity_dict)
write_dict("../data/WK3l-15k_EN/relation_dict.txt", relation_dict)

In [ ]:
df = df.drop_duplicates()
df.to_csv("../data/WK3l-15k_EN/triple_id.txt", sep="\t", index=False, header=None)

In [ ]:
import random

random.seed(0)

indices = list(range(0, df.shape[0]))
random.shuffle(indices)
valid_size = test_size = df.shape[0] // 5
train_size = df.shape[0] - valid_size - test_size

train = df.iloc[indices[:train_size]]
valid = df.iloc[indices[train_size:train_size+valid_size]]
test = df.iloc[indices[train_size+valid_size:]]


In [ ]:
train.to_csv("../data/WK3l-15k_EN/train_triple_id.txt", sep="\t", index=False, header=None)
valid.to_csv("../data/WK3l-15k_EN/valid_triple_id.txt", sep="\t", index=False, header=None)
test.to_csv("../data/WK3l-15k_EN/test_triple_id.txt", sep="\t", index=False, header=None)

In [ ]:
teacher_dict = read_dict("../data/CN3l_EN/entity_dict.txt")
student_dict = read_dict("../data/CN3l_FR/entity_dict.txt")

In [ ]:
df1 = pd.read_csv("~/Downloads/data/CN3l/en_fr/en2fr_cn.csv", sep="@@@", header=None)
df1.columns = ["teacher", "student"]
df2 = pd.read_csv("~/Downloads/data/CN3l/en_fr/fr2en_cn.csv", sep="@@@", header=None)
df2.columns = ["student", "teacher"]

In [ ]:
df1["teacher"] = df1["teacher"].apply(lambda x: teacher_dict.get(x, -1))
df1["student"] = df1["student"].apply(lambda x: student_dict.get(x, -1))
df2["teacher"] = df2["teacher"].apply(lambda x: teacher_dict.get(x, -1))
df2["student"] = df2["student"].apply(lambda x: student_dict.get(x, -1))

In [ ]:
df2

In [ ]:
df1 = pd.DataFrame([row for row in df1.drop_duplicates().itertuples() if row[1] != -1 and row[2] != -1]).drop(columns="Index")
df2 = pd.DataFrame([row for row in df2.drop_duplicates().itertuples() if row[1] != -1 and row[2] != -1]).drop(columns="Index")

In [ ]:
df1

In [ ]:
df2

In [ ]:
len(set(df1["teacher"]) & set(df2["teacher"])), len(set(df1["student"]) & set(df2["student"]))

In [ ]:
df = pd.merge(df1, df2, how="inner")

In [ ]:
df

In [ ]:
df1.to_csv("cn3l_en_fr_aligned_entity_id.txt", sep="\t", index=False, header=None)

In [ ]:
e_dict1 = read_dict("../data/DBP_DB_W/entity_dict.txt")
e_dict2 = read_dict("../data/DBP_DB_Y/entity_dict.txt")
r_dict1 = read_dict("../data/DBP_DB_W/relation_dict.txt")
r_dict2 = read_dict("../data/DBP_DB_Y/relation_dict.txt")

In [ ]:
e_dict = {x: i for i, x in enumerate(sorted(set(e_dict1.keys()) | set(e_dict2.keys())))}
r_dict = {x: i for i, x in enumerate(sorted(set(r_dict1.keys()) | set(r_dict2.keys())))}

In [ ]:
len(e_dict), len(r_dict)

In [ ]:
write_dict("../data/DBP_DB/entity_dict.txt", e_dict)
write_dict("../data/DBP_DB/relation_dict.txt", r_dict)

In [ ]:
e_dict1_rev = {v: k for k, v in e_dict1.items()}
e_dict2_rev = {v: k for k, v in e_dict2.items()}
r_dict1_rev = {v: k for k, v in r_dict1.items()}
r_dict2_rev = {v: k for k, v in r_dict2.items()}

In [ ]:
df1 = pd.read_csv("../data/DBP_DB_W/test_triple_id.txt", header=None, sep="\t")
df2 = pd.read_csv("../data/DBP_DB_Y/test_triple_id.txt", header=None, sep="\t")
df1.columns = ["head", "relation", "tail"]
df2.columns = ["head", "relation", "tail"]

In [ ]:
df1["head"] = df1["head"].apply(lambda x: e_dict[e_dict1_rev[x]])
df1["relation"] = df1["relation"].apply(lambda x: r_dict[r_dict1_rev[x]])
df1["tail"] = df1["tail"].apply(lambda x: e_dict[e_dict1_rev[x]])
df2["head"] = df2["head"].apply(lambda x: e_dict[e_dict2_rev[x]])
df2["relation"] = df2["relation"].apply(lambda x: r_dict[r_dict2_rev[x]])
df2["tail"] = df2["tail"].apply(lambda x: e_dict[e_dict2_rev[x]])

In [ ]:
df = pd.concat([df1, df2], axis=0)

In [ ]:
df.to_csv("../data/DBP_DB/test_triple_id.txt", sep="\t", index=False, header=None)

In [ ]:
teacher_dict = read_dict("../data/DBP_DB/entity_dict.txt")
student_dict = read_dict("../data/DBP_WD/entity_dict.txt")

In [ ]:
df1 = pd.read_csv("../data/SHARED/dbp_w_wd_aligned_entity_id.txt", sep="\t", header=None)
df1.columns = ["teacher", "student"]
df2 = pd.read_csv("../data/SHARED/dbp_y_yg_aligned_entity_id.txt", sep="\t", header=None)
df2.columns = ["teacher", "student"]

In [ ]:
df1["teacher"] = df1["teacher"].apply(lambda x: e_dict2.get(e_dict1_rev[x], -1))
df2["teacher"] = df2["teacher"].apply(lambda x: e_dict1.get(e_dict2_rev[x], -1))

In [ ]:
df1 = pd.DataFrame([row for row in df1.drop_duplicates().itertuples() if row[1] != -1]).drop(columns="Index")
df2 = pd.DataFrame([row for row in df2.drop_duplicates().itertuples() if row[1] != -1]).drop(columns="Index")

In [ ]:
df1.shape

In [ ]:
df2.shape

In [ ]:
df1.to_csv("../data/SHARED/dbp_y_wd_aligned_entity_id.txt", sep="\t", index=False, header=None)
df2.to_csv("../data/SHARED/dbp_w_yg_aligned_entity_id.txt", sep="\t", index=False, header=None)

In [ ]:
# student
# ent1 = [line.strip().lower().split(" ", 1)[1][1:-4].split() for line in open("../data/FB15k-237/name_en_full.txt")] 
ent1 = [line.strip().lower().split("\t", 1)[0].split("_") for line in open("../data/WK3l-15k_EN_F/entity_dict.txt")]

In [ ]:
# teacher
# ent2 = [line.strip().lower().split(" ", 1)[1][1:-4].split() for line in open("../data/FB15k-237/name_en_full.txt")] 
# ent2 = [line.strip().lower().split("\t", 1)[0].split("_") for line in open("../data/WK3l-15k_EN_F/entity_dict.txt")]
ent2 = [line.strip().lower().split("\t", 1)[0][28:].split("_") for line in open("../data/DBP_DB_W/entity_dict.txt")]

In [ ]:
print(len(ent1), len(ent2))

In [ ]:
def simpson_sim(w1, w2):
    w1 = set(w1)
    w2 = set(w2)
    if len(w1) == 0 and len(w2) == 0:
        return 0
    if len(w1) == 0 or len(w2) == 0:
        return 0
    return len(w1 & w2) / min(len(w1), len(w2))

In [ ]:
from tqdm import tqdm
import bisect
sims = []
ent1.sort()
ent2.sort()
ent1_str = [" ".join(e1) for e1 in ent1]
ent2_str = [" ".join(e2) for e2 in ent2]
for j, e2 in tqdm(enumerate(ent2)):
    max_i = 0
    max_sim = -1
    i = bisect.bisect_left(ent1_str, e2[0])
    while i < len(ent1):
        e1 = ent1[i]
        if e1[0] != e2[0]:
           break 
        sim = simpson_sim(e1, e2)
        if sim > max_sim:
            max_sim = sim
            max_i = i
        i += 1
    sims.append((max_i, j, max_sim))

In [ ]:
sims.sort(key=lambda x: x[2], reverse=True)

In [ ]:
ent12 = []
for i, j, sim in sims:
    if sim < 0.9:
        break
    ent12.append((i, j))

In [ ]:
print(len(ent12), len(ent1), len(ent2))

In [ ]:
len(set([x[0] for x in ent12])), len(set([x[1] for x in ent12]))

In [ ]:
with open("../data/SHARED/dbp_w_wk3l-15k_en_f_aligned_entity_id.txt", "w") as f:
    for x in ent12:
        f.write("%d\t%d\n" % (x[1], x[0]))

In [ ]:
with open("../data/SHARED/wk3l-15k_en_f_fr_aligned_entity_id.txt") as f:
    lines = f.readlines()
with open("../data/SHARED/wk3l-15k_fr_en_f_aligned_entity_id.txt", "w") as f:
    for line in lines:
        line = line.split()
        f.write("%s\t%s\n" % (line[1], line[0]))

In [ ]:
df1 = pd.read_csv("../data/SHARED/wk3l-15k_fr_en_f_aligned_entity_id.txt", header=None, sep="\t")
df2 = pd.read_csv("../data/SHARED/dbp_w_wk3l-15k_en_f_aligned_entity_id.txt", header=None, sep="\t")
df3 = pd.read_csv("../data/SHARED/fb15k_wk3l-15k_en_f_aligned_entity_id.txt", header=None, sep="\t")

In [ ]:
df1.columns = ["teacher", "student"]
df2.columns = ["teacher", "student"]
df3.columns = ["teacher", "student"]

In [ ]:
print(df1.shape, df2.shape, df3.shape)

In [ ]:
df = pd.concat([df1, df2, df3], axis=0)
print(df.shape)

In [ ]:
len(set(df["student"]))